In [ ]:
!pip install pandas tqdm

In [1]:
from google.colab import files
import pandas as pd

# завантаження локального файлу
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"Uploaded file: {filename}")

# читаємо CSV
df = pd.read_csv(filename)
print(f"Rows loaded: {len(df)}")
df.head(3)

Saving processed_v2.csv to processed_v2.csv
Uploaded file: processed_v2.csv
Rows loaded: 109166


,split,sent_id,token_id,token,lemma,upos,head,deprel
0,train,ParlaMint-UA_2003-10-14-m0.u1.p1.lang1.s1,1,Доброго,добрий,ADJ,2,amod
1,train,ParlaMint-UA_2003-10-14-m0.u1.p1.lang1.s1,2,ранку,ранок,NOUN,0,root
2,train,ParlaMint-UA_2003-10-14-m0.u1.p1.lang1.s1,3,",",",",PUNCT,6,punct


In [2]:
import os
import json

os.makedirs("resources", exist_ok=True)

# Місяці українською
months_ua = ["січня","лютого","березня","квітня","травня","червня",
             "липня","серпня","вересня","жовтня","листопада","грудня"]
with open("resources/months_ua.txt","w",encoding="utf-8") as f:
    f.write("\n".join(months_ua))

# Валюти
currencies = {"UAH":["грн","₴","UAH"], "USD":["$", "USD"], "EUR":["€","EUR"]}
with open("resources/currencies.json","w",encoding="utf-8") as f:
    json.dump(currencies,f, ensure_ascii=False, indent=2)

print("Resources created: months_ua.txt, currencies.json")

Resources created: months_ua.txt, currencies.json


In [3]:
import json

os.makedirs("tests", exist_ok=True)

ie_edge_cases = [
    {"id":1,"raw_text":"Зустріч 25 грудня 2024","field_type":"DATE","expected_behavior":"DATE parsed as 2024-12-25"},
    {"id":2,"raw_text":"Сума 1 000 грн","field_type":"AMOUNT","expected_behavior":"AMOUNT=1000, currency=UAH"},
    {"id":3,"raw_text":"Договір №123/2024","field_type":"DOC_ID","expected_behavior":"DOC_ID=123/2024"},
    {"id":4,"raw_text":"Телефон +380501234567","field_type":"PHONE","expected_behavior":"PHONE normalized +380501234567"},
    {"id":5,"raw_text":"Дата 01.01.2023","field_type":"DATE","expected_behavior":"DATE=2023-01-01"},
    # додаємо ще до 20+ кейсів
]

with open("tests/ie_edge_cases.jsonl","w",encoding="utf-8") as f:
    for case in ie_edge_cases:
        f.write(json.dumps(case, ensure_ascii=False)+"\n")
print("Edge cases saved: tests/ie_edge_cases.jsonl")

Edge cases saved: tests/ie_edge_cases.jsonl


In [4]:
os.makedirs("data/sample", exist_ok=True)

gold_subset = [
    {
        "text_id":"1",
        "text":"Зустріч 25 грудня 2024, сума 1 000 грн, договір №123/2024",
        "field_type":"DATE",
        "span_text":"25 грудня 2024",
        "start_char":9,
        "end_char":25,
        "normalized_value":"2024-12-25"
    },
    {
        "text_id":"1",
        "text":"Зустріч 25 грудня 2024, сума 1 000 грн, договір №123/2024",
        "field_type":"AMOUNT",
        "span_text":"1 000 грн",
        "start_char":27,
        "end_char":35,
        "normalized_value":"1000 UAH"
    },
    {
        "text_id":"1",
        "text":"Зустріч 25 грудня 2024, сума 1 000 грн, договір №123/2024",
        "field_type":"DOC_ID",
        "span_text":"№123/2024",
        "start_char":37,
        "end_char":46,
        "normalized_value":"123/2024"
    }
]

with open("data/sample/lab4_gold_ie.jsonl","w",encoding="utf-8") as f:
    for row in gold_subset:
        f.write(json.dumps(row, ensure_ascii=False)+"\n")
print("Gold subset saved: data/sample/lab4_gold_ie.jsonl")

Gold subset saved: data/sample/lab4_gold_ie.jsonl


In [5]:
import re
from tqdm import tqdm

def extract_dates(text):
    results=[]
    # простий regex на DD/MM/YYYY або DD місяць YYYY
    date_regex = r"(\d{1,2})[\.\-/ ]?(січня|лютого|березня|квітня|травня|червня|липня|серпня|вересня|жовтня|листопада|грудня)[\.\-/ ]?(\d{4})"
    for m in re.finditer(date_regex,text):
        day, month, year = m.groups()
        month_idx = months_ua.index(month)+1
        norm = f"{year}-{month_idx:02d}-{int(day):02d}"
        results.append({"field_type":"DATE","value":norm,"start_char":m.start(),"end_char":m.end(),"method":"regex_date"})
    return results

def extract_amounts(text):
    results=[]
    amount_regex = r"(\d[\d\s]*)(грн|\$|USD|EUR|€|₴|UAH)"
    for m in re.finditer(amount_regex,text):
        val, cur = m.groups()
        val = float(val.replace(" ",""))
        cur_norm = None
        for k,v in currencies.items():
            if cur in v:
                cur_norm=k
        results.append({"field_type":"AMOUNT","value":f"{val} {cur_norm if cur_norm else cur}","start_char":m.start(),"end_char":m.end(),"method":"regex_amount"})
    return results

def extract_doc_ids(text):
    results=[]
    doc_regex = r"№\s*\d+(/\d+)?"
    for m in re.finditer(doc_regex,text):
        results.append({"field_type":"DOC_ID","value":m.group(),"start_char":m.start(),"end_char":m.end(),"method":"regex_docid"})
    return results

def extract_all(text):
    return extract_dates(text)+extract_amounts(text)+extract_doc_ids(text)

In [6]:
os.makedirs("output", exist_ok=True)
all_results = []

for idx,row in tqdm(df.iterrows(), total=len(df)):
    text_id = row.get("sent_id", str(idx))
    text = row.get("processed_text","")
    results = extract_all(text)
    for r in results:
        r["text_id"]=text_id
        all_results.append(r)

with open("output/lab4_ie_results.jsonl","w",encoding="utf-8") as f:
    for r in all_results:
        f.write(json.dumps(r, ensure_ascii=False)+"\n")

print(f"IE results saved: {len(all_results)} extractions → output/lab4_ie_results.jsonl")

100%|██████████| 109166/109166 [00:07<00:00, 15438.47it/s]

IE results saved: 0 extractions → output/lab4_ie_results.jsonl


In [7]:
import os
os.makedirs("docs", exist_ok=True)

policy_text = """
Політика Rule-based IE
=====================

Типи полів для витягу:
1. DATE (ДАТА)
   - Формат: YYYY-MM-DD
   - Джерело: дати у processed_text, що відповідають формату DD <місяць> YYYY
2. AMOUNT (СУМА)
   - Формат: float + валюта (UAH/USD/EUR)
   - Джерело: числа, за якими йде символ або назва валюти
3. DOC_ID (НОМЕР ДОКУМЕНТА)
   - Формат: raw (наприклад, №123/2024)
   - Джерело: № + цифри + опційно /цифри
"""

with open("docs/ie_policy.md","w",encoding="utf-8") as f:
    f.write(policy_text)

print("Політика збережена: docs/ie_policy.md")

Політика збережена: docs/ie_policy.md


In [8]:
summary_text = f"""
Лабораторна робота 4 — Аудит витягу інформації
==============================================

Загальна кількість рядків: {len(df)}
Загальна кількість витягів: {len(all_results)}

Кількість витягів по типах полів:
DATE (ДАТА): {sum(1 for r in all_results if r['field_type']=='DATE')}
AMOUNT (СУМА): {sum(1 for r in all_results if r['field_type']=='AMOUNT')}
DOC_ID (НОМЕР ДОКУМЕНТА): {sum(1 for r in all_results if r['field_type']=='DOC_ID')}

Коментар:
- Витяг базується на простих правилах regex та словниках.
- Precision та recall слід оцінювати на gold subset.
- Для подальшого поліпшення можна додати більше правил або контекстні анти-правила.
"""

with open("docs/audit_summary_lab4.md","w",encoding="utf-8") as f:
    f.write(summary_text)

print("Аудит збережено: docs/audit_summary_lab4.md")

Аудит збережено: docs/audit_summary_lab4.md
